# 02 — Preprocessing

This notebook performs the documented data-quality filtering, invalid-value handling, target-outlier treatment, transformation, missing-value treatment, predictor clipping, encoding, and scaling. It writes the cleaned dataset to `data/processed/`.

### Imports — why this cell is needed
Loads the preprocessing and validation libraries used in this notebook.

In [ ]:
import re
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)


from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.feature_selection import SelectPercentile, f_regression
from sklearn.decomposition import TruncatedSVD
from sklearn.model_selection import (
    GroupShuffleSplit,
    GroupKFold,
    cross_validate,
    GridSearchCV,
    RandomizedSearchCV
)
from sklearn.linear_model import Ridge
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    mean_squared_log_error,
    r2_score
)

### Repository paths — why this cell is needed
Keeps all file access relative to the repository.

In [ ]:
def find_repo_root():
    current = Path.cwd().resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "data").exists() and (candidate / "notebooks").exists():
            return candidate
    raise FileNotFoundError(
        "Repository root not found. Run this notebook from inside the cloned "
        "CSE437 repository, with data/ and notebooks/ folders present."
    )

ROOT = find_repo_root()
RAW_DIR = ROOT / "data" / "raw"
PROCESSED_DIR = ROOT / "data" / "processed"
FIGURES_DIR = ROOT / "figures"
MODELS_DIR = ROOT / "models"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR.mkdir(parents=True, exist_ok=True)

print("Repository root:", ROOT)

### Load the untouched raw dataset — why this cell is needed
Preprocessing starts from the original file in `data/raw/`.

In [ ]:
csv_files = sorted(RAW_DIR.glob("*.csv"))
if not csv_files:
    raise FileNotFoundError(
        "No CSV was found in data/raw/. Put the original Seattle CSV there first."
    )

RAW_CSV = csv_files[0]
df_raw = pd.read_csv(RAW_CSV, low_memory=False)

print("Loaded:", RAW_CSV.name)
print("Raw shape:", df_raw.shape)
display(df_raw.head())

### Resolve columns and leakage fields — why this cell is needed
Recreates the exact safe-column mapping used by the final project.

In [ ]:
def normalize_name(name):
    return re.sub(r"[^a-z0-9]", "", str(name).lower())

norm_map = {normalize_name(c): c for c in df_raw.columns}

def find_column(aliases, required=False):
    for alias in aliases:
        key = normalize_name(alias)
        if key in norm_map:
            return norm_map[key]
    if required:
        raise KeyError(
            f"Required column not found. Tried: {aliases}\n"
            f"Available columns: {list(df_raw.columns)}"
        )
    return None

TARGET_RAW = find_column(
    ["SiteEUI(kBtu/sf)", "SiteEUI", "siteeui_kbtu_sf"],
    required=True
)
GROUP_RAW = find_column(
    ["OSEBuildingID", "OSE Building ID", "osebuildingid"],
    required=True
)

COMPLIANCE_RAW = find_column(
    ["ComplianceStatus", "compliancestatus"]
)
COMPLIANCE_ISSUE_RAW = find_column(
    ["ComplianceIssue", "complianceissue"]
)
DEMOLISHED_RAW = find_column(
    ["Demolished", "demolished"]
)

FEATURE_ALIASES = {
    "DataYear": ["DataYear", "datayear"],
    "BuildingType": ["BuildingType", "buildingtype"],
    "EPAPropertyType": ["EPAPropertyType", "epapropertytype"],
    "LargestPropertyUseType": [
        "LargestPropertyUseType", "largestpropertyusetype"
    ],
    "SecondLargestPropertyUseType": [
        "SecondLargestPropertyUseType", "secondlargestpropertyusetype"
    ],
    "ThirdLargestPropertyUseType": [
        "ThirdLargestPropertyUseType", "thirdlargestpropertyusetype"
    ],
    "Neighborhood": ["Neighborhood", "neighborhood"],
    "CouncilDistrictCode": [
        "CouncilDistrictCode", "councildistrictcode"
    ],
    "ZipCode": ["ZipCode", "zipcode"],
    "YearBuilt": ["YearBuilt", "yearbuilt"],
    "NumberofFloors": ["NumberofFloors", "numberoffloors"],
    "NumberofBuildings": ["NumberofBuildings", "numberofbuildings"],
    "PropertyGFATotal": ["PropertyGFATotal", "propertygfatotal"],
    "PropertyGFABuilding": [
        "PropertyGFABuilding(s)", "PropertyGFABuilding",
        "propertygfabuildings"
    ],
    "PropertyGFAParking": [
        "PropertyGFAParking", "propertygfaparking"
    ],
    "LargestPropertyUseTypeGFA": [
        "LargestPropertyUseTypeGFA", "largestpropertyusetypegfa"
    ],


    "SecondLargestPropertyUseTypeGFA": [
        "SecondLargestPropertyUseTypeGFA",
        "secondlargestpropertyuse"
    ],
    "ThirdLargestPropertyUseTypeGFA": [
        "ThirdLargestPropertyUseTypeGFA",
        "thirdlargestpropertyusetypegfa"
    ],
    "Latitude": ["Latitude", "latitude"],
    "Longitude": ["Longitude", "longitude"]
}

resolved_features = {}
for clean_name, aliases in FEATURE_ALIASES.items():
    raw_name = find_column(aliases)
    if raw_name is not None:
        resolved_features[clean_name] = raw_name

LEAKAGE_PATTERNS = [
    "energystar",
    "siteeui",
    "sourceeui",
    "siteenergyuse",
    "electricity",
    "naturalgas",
    "steamuse",
    "otherfuel",
    "ghg",
    "emission"
]

leakage_columns = [
    c for c in df_raw.columns
    if any(pattern in normalize_name(c) for pattern in LEAKAGE_PATTERNS)
]

print("Target:", TARGET_RAW)
print("Building grouping ID:", GROUP_RAW)
print("Compliance status:", COMPLIANCE_RAW)
print("Compliance issue:", COMPLIANCE_ISSUE_RAW)
print("Demolished flag:", DEMOLISHED_RAW)

print("\nLeakage-related columns found and excluded from predictors:")
for col in leakage_columns:
    print(" -", col)

print("\nSafe predictor columns resolved:")
for clean_name, raw_name in resolved_features.items():
    print(f" - {clean_name} <- {raw_name}")


for raw_name in resolved_features.values():
    assert not any(
        pattern in normalize_name(raw_name)
        for pattern in LEAKAGE_PATTERNS
    ), f"Leakage predictor detected: {raw_name}"

### Apply documented data-quality exclusions — why this cell is needed
Removes records explicitly marked Not Compliant or Demolished.

In [ ]:
quality_mask = pd.Series(True, index=df_raw.index)

if COMPLIANCE_RAW is not None:
    compliance_clean = (
        df_raw[COMPLIANCE_RAW]
        .astype("string")
        .str.strip()
        .str.lower()
    )

    print("Compliance status before filtering:")
    display(compliance_clean.value_counts(dropna=False).to_frame("Count"))

    noncompliant_mask = compliance_clean.eq("not compliant")
    quality_mask &= ~noncompliant_mask
    print("Not Compliant rows excluded:", int(noncompliant_mask.sum()))

if DEMOLISHED_RAW is not None:
    demolished_clean = (
        df_raw[DEMOLISHED_RAW]
        .astype("string")
        .str.strip()
        .str.lower()
        .isin(["true", "1", "yes"])
    )

    quality_mask &= ~demolished_clean
    print("Demolished rows excluded:", int(demolished_clean.sum()))

df_quality = df_raw.loc[quality_mask].copy()

print("Rows before quality filtering:", len(df_raw))
print("Rows after quality filtering:", len(df_quality))
print("Rows removed for documented quality reasons:",
      len(df_raw) - len(df_quality))

### Build the safe modelling table — why this cell is needed
Copies only approved, non-leakage predictors plus the target and grouping ID.

In [ ]:
df = pd.DataFrame(index=df_quality.index)

df["OSEBuildingID"] = df_quality[GROUP_RAW].astype("string")
df["SiteEUI"] = pd.to_numeric(
    df_quality[TARGET_RAW],
    errors="coerce"
)

for clean_name, raw_name in resolved_features.items():
    df[clean_name] = df_quality[raw_name]

numeric_expected = [
    "DataYear",
    "YearBuilt",
    "NumberofFloors",
    "NumberofBuildings",
    "PropertyGFATotal",
    "PropertyGFABuilding",
    "PropertyGFAParking",
    "LargestPropertyUseTypeGFA",
    "SecondLargestPropertyUseTypeGFA",
    "ThirdLargestPropertyUseTypeGFA",
    "Latitude",
    "Longitude"
]

for col in numeric_expected:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

for col in [
    "BuildingType",
    "EPAPropertyType",
    "LargestPropertyUseType",
    "SecondLargestPropertyUseType",
    "ThirdLargestPropertyUseType",
    "Neighborhood",
    "CouncilDistrictCode",
    "ZipCode"
]:
    if col in df.columns:

        df[col] = df[col].astype(object)
        df[col] = df[col].where(pd.notna(df[col]), np.nan)

before_supervised_cleaning = len(df)
missing_target = df["SiteEUI"].isna().sum()
nonpositive_target = (df["SiteEUI"] <= 0).sum()
missing_group = df["OSEBuildingID"].isna().sum()

df = df[
    df["SiteEUI"].notna()
    & (df["SiteEUI"] > 0)
    & df["OSEBuildingID"].notna()
].copy()

duplicates = df.duplicated().sum()
df = df.drop_duplicates().reset_index(drop=True)

print("Rows before supervised cleaning:", before_supervised_cleaning)
print("Missing target rows removed:", int(missing_target))
print("Non-positive target rows removed:", int(nonpositive_target))
print("Missing building-ID rows removed:", int(missing_group))
print("Exact duplicates removed:", int(duplicates))
print("Usable rows:", len(df))
print("Unique buildings:", df["OSEBuildingID"].nunique())

### Correct impossible structural values — why this cell is needed
Turns impossible measurements into missing values so they can be imputed consistently.

In [ ]:
invalid_changes = []

def replace_invalid_with_nan(column, invalid_mask, reason):
    if column not in df.columns:
        return

    count = int(invalid_mask.fillna(False).sum())

    invalid_changes.append({
        "Feature": column,
        "Invalid values changed to NaN": count,
        "Reason": reason
    })

    df.loc[invalid_mask.fillna(False), column] = np.nan

if "NumberofFloors" in df.columns:
    replace_invalid_with_nan(
        "NumberofFloors",
        df["NumberofFloors"] <= 0,
        "A building cannot have zero or negative floors"
    )

if "NumberofBuildings" in df.columns:
    replace_invalid_with_nan(
        "NumberofBuildings",
        df["NumberofBuildings"] <= 0,
        "A property record cannot contain zero or negative buildings"
    )

if "PropertyGFATotal" in df.columns:
    replace_invalid_with_nan(
        "PropertyGFATotal",
        df["PropertyGFATotal"] <= 0,
        "Total floor area must be positive"
    )

if "PropertyGFABuilding" in df.columns:
    replace_invalid_with_nan(
        "PropertyGFABuilding",
        df["PropertyGFABuilding"] <= 0,
        "Building floor area must be positive"
    )

if "PropertyGFAParking" in df.columns:
    replace_invalid_with_nan(
        "PropertyGFAParking",
        df["PropertyGFAParking"] < 0,
        "Parking area cannot be negative"
    )

for col in [
    "LargestPropertyUseTypeGFA",
    "SecondLargestPropertyUseTypeGFA",
    "ThirdLargestPropertyUseTypeGFA"
]:
    if col in df.columns:
        replace_invalid_with_nan(
            col,
            df[col] < 0,
            "Use-type floor area cannot be negative"
        )

if {"YearBuilt", "DataYear"}.issubset(df.columns):
    invalid_year = (
        (df["YearBuilt"] < 1800)
        | (df["YearBuilt"] > df["DataYear"])
    )
    replace_invalid_with_nan(
        "YearBuilt",
        invalid_year,
        "Construction year must be plausible and not later than reporting year"
    )

invalid_value_evidence = pd.DataFrame(invalid_changes)
display(invalid_value_evidence)

### Inspect the target after quality cleaning — why this cell is needed
Shows the remaining target distribution before the objective target-outlier rule.

In [ ]:
print("SiteEUI descriptive statistics after quality cleaning:")
display(df["SiteEUI"].describe().to_frame().T)

print("\nHighest remaining SiteEUI values:")
display(
    df[
        [
            c for c in [
                "OSEBuildingID",
                "DataYear",
                "BuildingType",
                "EPAPropertyType",
                "PropertyGFATotal",
                "SiteEUI"
            ] if c in df.columns
        ]
    ]
    .sort_values("SiteEUI", ascending=False)
    .head(15)
)

print("\nSiteEUI percentiles after quality cleaning:")
display(
    df["SiteEUI"]
    .quantile([0.50, 0.90, 0.95, 0.99, 0.995, 0.999, 1.00])
    .to_frame("SiteEUI")
)

### Apply the objective 99.9th-percentile target rule — why this cell is needed
Removes only the most extreme 0.1% of target values using the fixed rule used in the final analysis.

In [ ]:
TARGET_OUTLIER_QUANTILE = 0.999

rows_before_target_filter = len(df)
target_upper_limit = df["SiteEUI"].quantile(TARGET_OUTLIER_QUANTILE)

target_outlier_mask = df["SiteEUI"] > target_upper_limit
target_outliers = (
    df.loc[target_outlier_mask]
    .sort_values("SiteEUI", ascending=False)
    .copy()
)

print(f"Target outlier rule: SiteEUI > {TARGET_OUTLIER_QUANTILE:.1%} percentile")
print(f"99.9th-percentile threshold: {target_upper_limit:.3f} kBtu/sf")
print("Rows before target-outlier treatment:", rows_before_target_filter)
print("Extreme target rows removed:", int(target_outlier_mask.sum()))
print(
    "Percentage removed:",
    round(target_outlier_mask.mean() * 100, 4),
    "%"
)

show_cols = [
    c for c in [
        "OSEBuildingID",
        "DataYear",
        "BuildingType",
        "EPAPropertyType",
        "PropertyGFATotal",
        "SiteEUI"
    ] if c in target_outliers.columns
]

print("\nRemoved extreme target observations:")
display(target_outliers[show_cols])

target_before_after = pd.DataFrame({
    "Before target-outlier treatment": [
        df["SiteEUI"].min(),
        df["SiteEUI"].median(),
        df["SiteEUI"].mean(),
        df["SiteEUI"].max(),
        df["SiteEUI"].skew()
    ]
}, index=["Minimum", "Median", "Mean", "Maximum", "Skewness"])

df = df.loc[~target_outlier_mask].reset_index(drop=True)

target_before_after["After target-outlier treatment"] = [
    df["SiteEUI"].min(),
    df["SiteEUI"].median(),
    df["SiteEUI"].mean(),
    df["SiteEUI"].max(),
    df["SiteEUI"].skew()
]

print("\nEvidence of what changed:")
display(target_before_after)

print("Rows remaining for modelling:", len(df))

### Transformation evidence — why this cell is needed
Demonstrates the effect of `log1p` on SiteEUI skewness and saves the report figures.

In [ ]:
raw_skew = df["SiteEUI"].skew()
log_skew = np.log1p(df["SiteEUI"]).skew()

print("Raw SiteEUI skewness:", round(raw_skew, 3))
print("log1p(SiteEUI) skewness:", round(log_skew, 3))

plt.figure(figsize=(8, 4))
plt.hist(df["SiteEUI"], bins=60)
plt.xlabel("SiteEUI")
plt.ylabel("Frequency")
plt.title("SiteEUI Before Transformation")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "siteeui_before_transformation.png", dpi=200, bbox_inches="tight")
plt.show()

plt.figure(figsize=(8, 4))
plt.hist(np.log1p(df["SiteEUI"]), bins=60)
plt.xlabel("log1p(SiteEUI)")
plt.ylabel("Frequency")
plt.title("SiteEUI After log1p Transformation")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "siteeui_after_log1p.png", dpi=200, bbox_inches="tight")
plt.show()

### Save the cleaned dataset — why this cell is needed
Creates the processed file used by the next notebook without altering the original raw CSV.

In [ ]:
CLEANED_CSV = PROCESSED_DIR / "seattle_energy_cleaned.csv"
df.to_csv(CLEANED_CSV, index=False)
print("Saved:", CLEANED_CSV.relative_to(ROOT))
print("Shape:", df.shape)

### Grouped split for preprocessing evidence — why this cell is needed
Fits preprocessing only on development buildings so the before/after evidence is leakage-safe.

In [ ]:
TARGET = "SiteEUI"
GROUP = "OSEBuildingID"

X = df.drop(columns=[TARGET, GROUP])
y = df[TARGET]
groups = df[GROUP]

splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=RANDOM_STATE
)

dev_idx, test_idx = next(
    splitter.split(X, y, groups=groups)
)

X_dev = X.iloc[dev_idx].reset_index(drop=True)
y_dev = y.iloc[dev_idx].reset_index(drop=True)
groups_dev = groups.iloc[dev_idx].reset_index(drop=True)

X_test = X.iloc[test_idx].reset_index(drop=True)
y_test = y.iloc[test_idx].reset_index(drop=True)
groups_test = groups.iloc[test_idx].reset_index(drop=True)

shared_buildings = set(groups_dev).intersection(set(groups_test))

print("Development rows:", len(X_dev))
print("Test rows:", len(X_test))
print("Development buildings:", groups_dev.nunique())
print("Test buildings:", groups_test.nunique())
print("Shared building IDs:", len(shared_buildings))

assert len(shared_buildings) == 0

### Define missing-value, outlier, encoding, and scaling preprocessing — why this cell is needed
Recreates the final pipeline's preprocessing rules.

In [ ]:
numeric_features = X_dev.select_dtypes(include=np.number).columns.tolist()
categorical_features = [
    c for c in X_dev.columns
    if c not in numeric_features
]


for col in categorical_features:
    X_dev[col] = X_dev[col].astype(object)
    X_test[col] = X_test[col].astype(object)
    X_dev[col] = X_dev[col].where(pd.notna(X_dev[col]), np.nan)
    X_test[col] = X_test[col].where(pd.notna(X_test[col]), np.nan)

class QuantileClipper(BaseEstimator, TransformerMixin):
    def __init__(self, columns=None, lower=0.01, upper=0.99):
        self.columns = columns
        self.lower = lower
        self.upper = upper

    def fit(self, X, y=None):
        X = X.copy()
        self.bounds_ = {}
        for col in self.columns or []:
            series = pd.to_numeric(X[col], errors="coerce")
            self.bounds_[col] = (
                series.quantile(self.lower),
                series.quantile(self.upper)
            )
        return self

    def transform(self, X):
        X = X.copy()
        for col, (low, high) in self.bounds_.items():
            if col in X.columns:
                X[col] = pd.to_numeric(
                    X[col], errors="coerce"
                ).clip(low, high)
        return X

def make_preprocessor():
    numeric_pipeline = Pipeline([
        ("imputer", SimpleImputer(
            strategy="median",
            keep_empty_features=True
        )),
        ("scaler", StandardScaler())
    ])

    categorical_pipeline = Pipeline([
        ("imputer", SimpleImputer(
            strategy="most_frequent",
            keep_empty_features=True
        )),
        ("onehot", OneHotEncoder(
            handle_unknown="ignore",
            min_frequency=10,
            sparse_output=False
        ))
    ])

    return ColumnTransformer([
        ("numeric", numeric_pipeline, numeric_features),
        ("categorical", categorical_pipeline, categorical_features)
    ], remainder="drop")

print("Numeric predictors:", len(numeric_features))
print("Categorical predictors:", len(categorical_features))
print("Missing development cells before preprocessing:",
      int(X_dev.isna().sum().sum()))

### Outlier-treatment evidence — why this cell is needed
Shows how numeric predictor extremes change after training-derived 1st/99th percentile clipping.

In [ ]:
clip_demo = QuantileClipper(
    columns=numeric_features,
    lower=0.01,
    upper=0.99
).fit(X_dev)

X_clip_demo = clip_demo.transform(X_dev)

outlier_evidence = []

for col in numeric_features:
    before = pd.to_numeric(X_dev[col], errors="coerce")
    after = pd.to_numeric(X_clip_demo[col], errors="coerce")

    changed = (
        before.notna()
        & after.notna()
        & (before != after)
    ).sum()

    outlier_evidence.append({
        "Feature": col,
        "Values Clipped": int(changed),
        "Minimum Before": before.min(),
        "Minimum After": after.min(),
        "Maximum Before": before.max(),
        "Maximum After": after.max()
    })

outlier_evidence = pd.DataFrame(outlier_evidence)
display(outlier_evidence)

### Imputation and scaling evidence — why this cell is needed
Shows missing values are removed and numeric features are standardized.

In [ ]:
preprocessor_demo = make_preprocessor()
X_prepared_demo = preprocessor_demo.fit_transform(
    X_clip_demo,
    y_dev
)

print("Missing cells before preprocessing:",
      int(X_dev.isna().sum().sum()))
print("NaN values after preprocessing:",
      int(np.isnan(X_prepared_demo).sum()))
print("Prepared matrix shape:", X_prepared_demo.shape)

numeric_demo = Pipeline([
    ("imputer", SimpleImputer(
        strategy="median",
        keep_empty_features=True
    )),
    ("scaler", StandardScaler())
])

scaled_numeric = numeric_demo.fit_transform(
    X_clip_demo[numeric_features]
)

scaling_evidence = pd.DataFrame({
    "Feature": numeric_features,
    "Mean Before": [
        pd.to_numeric(
            X_clip_demo[c], errors="coerce"
        ).mean()
        for c in numeric_features
    ],
    "Std Before": [
        pd.to_numeric(
            X_clip_demo[c], errors="coerce"
        ).std()
        for c in numeric_features
    ],
    "Mean After Scaling": np.mean(
        scaled_numeric, axis=0
    ),
    "Std After Scaling": np.std(
        scaled_numeric, axis=0
    )
})

display(scaling_evidence)